sklopit sve gradove u jedan csv pa ce se kasnije sve mergat

In [ ]:
# PRILOG_ANĐELA MAKSIMOVIĆ_VJETAR_04_29_2_253_26.xlsx

In [ ]:
import pandas as pd
file = "PRILOG_ANĐELA MAKSIMOVIĆ_VJETAR_04_29_2_253_26.xlsx"

xls = pd.ExcelFile(file)

print(xls.sheet_names)

In [ ]:
sheet = xls.sheet_names[0]

df = pd.read_excel(file, sheet_name=sheet)

print(df.head())
print(df.columns.tolist())
print(df.shape)

   Eg gh id Eg el abbreviation  Year  Month   Time  Val01  Val02  Val03  \
0  61MOST01              DD360  2020      1  00:00  309.0   41.0  323.0   
1  61MOST01               ff10  2020      1  00:00    2.8    5.3    2.1   
2  61MOST01              DD360  2020      1  00:10  322.0   39.0  330.0   
3  61MOST01               ff10  2020      1  00:10    3.1    5.6    2.1   
4  61MOST01              DD360  2020      1  00:20  316.0   51.0  343.0   

   Val04  Val05  ...  Val22  Val23  Val24  Val25  Val26  Val27  Val28  Val29  \
0  340.0  289.0  ...  356.0  328.0   21.0  310.0  337.0  292.0  278.0  218.0   
1    0.9    0.9  ...    1.9    1.5    1.9    0.3    0.7    1.2    1.1    1.9   
2  279.0  285.0  ...   21.0  310.0   38.0  333.0  309.0  259.0  272.0  215.0   
3    1.2    2.2  ...    2.1    1.6    3.6    0.7    0.6    1.3    0.5    2.2   
4  276.0  277.0  ...  340.0  287.0   43.0  301.0  295.0  266.0  249.0  212.0   

   Val30  Val31  
0  287.0  313.0  
1    1.3    1.0  
2  336.0  293.

In [ ]:
df = pd.read_excel(file, sheet_name=city)

long_df = df.melt(
    id_vars=["Year", "Month", "Time"],
    value_vars=[c for c in df.columns if c.startswith("Val")],
    var_name="day",
    value_name="wind_speed"
)

print(long_df.head())
print(long_df.shape)

In [ ]:
city_map = {
    "MOSTAR": "Mostar",
    "SANSKI MOST": "Sanski Most",
    "TUZLA": "Tuzla",
    "SARAJEVO": "Sarajevo",
    "BIHAĆ": "Bihac",
    "ZENICA": "Zenica",
    "LIVNO": "Livno",
    "BUGOJNO": "Bugojno",
    "GRADAČAC": "Gradacac"
}

In [ ]:
import pandas as pd

all_cities = []

sheets = pd.read_excel(file, sheet_name=None)

for city, df in sheets.items():

    city = city_map.get(city, city.title())

    df.columns = df.columns.astype(str).str.strip()

    val_cols = [c for c in df.columns if str(c).startswith("Val")]

    keep_cols = ["Year", "Month", "Time"] + val_cols
    df = df[keep_cols]

    print(f"\n=== {city} ===")
    print(f"Raw shape: {df.shape}")
    print(df[["Year", "Month", "Time"]].head(5))
    print(f"Time dtype: {df['Time'].dtype}")
    print(f"Sample Time values: {df['Time'].head(10).tolist()}")

    long_df = df.melt(
        id_vars=["Year", "Month", "Time"],
        value_vars=val_cols,
        var_name="day",
        value_name="wind_speed"
    )

    long_df = long_df.dropna(subset=["wind_speed"])
    long_df["day"] = long_df["day"].str.replace("Val", "", regex=False).astype(int)
    long_df["hour"] = long_df["Time"]

    final = long_df[["Year", "Month", "day", "hour", "wind_speed"]].copy()
    final.rename(columns={"Year": "year", "Month": "month"}, inplace=True)
    final["city"] = city

    print(f"Final shape: {final.shape}")
    all_cities.append(final)

wind_df = pd.concat(all_cities, ignore_index=True)

print(wind_df.head())
print(wind_df.shape)

wind_df.to_csv("fhz_windspeed.csv", index=False)


=== Mostar ===
Raw shape: (17280, 34)
   Year  Month   Time
0  2020      1  00:00
1  2020      1  00:00
2  2020      1  00:10
3  2020      1  00:10
4  2020      1  00:20
Time dtype: object
Sample Time values: ['00:00', '00:00', '00:10', '00:10', '00:20', '00:20', '00:30', '00:30', '00:40', '00:40']
Final shape: (503286, 6)

=== Sanski Most ===
Raw shape: (2304, 34)
   Year  Month   Time
0  2021      1  00:00
1  2021      1  00:00
2  2021      1  01:00
3  2021      1  01:00
4  2021      1  02:00
Time dtype: object
Sample Time values: ['00:00', '00:00', '01:00', '01:00', '02:00', '02:00', '03:00', '03:00', '04:00', '04:00']
Final shape: (69520, 6)

=== Tuzla ===
Raw shape: (17280, 34)
   Year  Month   Time
0  2020      1  00:00
1  2020      1  00:00
2  2020      1  00:10
3  2020      1  00:10
4  2020      1  00:20
Time dtype: object
Sample Time values: ['00:00', '00:00', '00:10', '00:10', '00:20', '00:20', '00:30', '00:30', '00:40', '00:40']
Final shape: (512840, 6)

=== Sarajevo ===
Ra

In [ ]:
print(long_df["hour"].iloc[0])
print(type(long_df["hour"].iloc[0]))

00:00
<class 'str'>


In [ ]:
wind_df["city"].unique()

array(['Mostar', 'Sanski Most', 'Tuzla', 'Sarajevo', 'Bihac', 'Zenica',
       'Livno', 'Bugojno', 'Gradacac'], dtype=object)